# Week 5 · Demo — The Critic-Creator Loop 🔁
### Two prompts that make an answer systematically better, round by round

You've learned to *judge* answers. Now the natural next move: use that same rubric not just to *score* an answer, but to **improve** it. The **Critic-Creator loop** is the first systematic prompt-improvement tool of the programme — and it's the shape of every reflection / self-refinement pattern you'll meet again in Weeks 11, 17, 22, and 28. Learn the shape once, here.

**The four boxes:** a **Creator** drafts → a **Critic** critiques it (using the same rubric as your judge) → the Creator **revises** → repeat until it's good enough → **Final**.

**How to use this notebook:** run top to bottom and watch a weak draft climb to a good answer across rounds. It runs **offline** (deterministic Creator/Critic stand-ins) or **live** (Creator = `gpt-4o-mini`, Critic = `gpt-4o`).

**Where this goes in your app:** the seed of `src/eval/critic_creator.py`.

---
### The key idea before we start
This is **two prompts, not one call.** One prompt plays *creator* (write/revise), a *separate* prompt plays *critic* (find what's wrong, using the rubric). Each round, the critic catches something specific the creator missed. It's the difference between *writing* an answer and *editing* one — and editing, done in a loop, is where quality comes from.

## 0 · Setup — a deliberately weak first draft

We start the Creator off with a **bad draft** — vague, missing both required facts — so you can watch the loop *fix* it. (In your app the first draft comes from `/ask`; here we hand-pick a weak one to make the improvement visible.)

In [ ]:
import os

USE_FAKE = True             # flip to False (+ OPENAI_API_KEY) for real Creator/Critic
CREATOR_MODEL = "gpt-4o-mini"   # the creator can be the cheap model
CRITIC_MODEL  = "gpt-4o"        # the critic should be the STRONG model (same as the judge)

QUESTION = "How many days can I work remotely, and what approval do I need?"
MUST = ["3 days", "manager approval"]     # the rubric's required facts
MAX_WORDS = 40

first_draft = "You can work from home sometimes if it's okay with people."
print("initial draft:", first_draft)

## 1 · The Critic — finds what's wrong, using the rubric

The Critic uses the **same rubric dimensions as your judge** (accuracy → required facts present; format → concise). But instead of a *score*, it returns **actionable issues** and a **`is_good_enough`** flag. That flag is what lets the loop stop on its own.

In [ ]:
def critic(answer):
    """Critique against the rubric. Returns issues + whether it's good enough to stop."""
    if USE_FAKE:
        issues = [f"missing required fact: '{m}'" for m in MUST if m.lower() not in answer.lower()]
        if len(answer.split()) > MAX_WORDS:
            issues.append(f"too long ({len(answer.split())} words); tighten to <= {MAX_WORDS}")
        return {"issues": issues, "is_good_enough": len(issues) == 0}
    return _real_critic(answer)

def _real_critic(answer):
    import json
    from openai import OpenAI
    client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
    tool = {"type": "function", "function": {"name": "critique", "parameters": {"type": "object",
        "properties": {"issues": {"type": "array", "items": {"type": "string"}},
                       "is_good_enough": {"type": "boolean"}},
        "required": ["issues", "is_good_enough"]}}}
    rubric = ("You are a strict critic. List concrete issues with the answer to this question: "
              f"{QUESTION!r}. It MUST state: {MUST}. It must be under {MAX_WORDS} words. "
              "If there are no issues, return an empty list and is_good_enough=true.")
    resp = client.chat.completions.create(model=CRITIC_MODEL, temperature=0,
        messages=[{"role": "system", "content": rubric},
                  {"role": "user", "content": answer}],
        tools=[tool], tool_choice={"type": "function", "function": {"name": "critique"}})
    return json.loads(resp.choices[0].message.tool_calls[0].function.arguments)

c0 = critic(first_draft)
print("issues found:", c0["issues"])
print("good enough? ", c0["is_good_enough"])

The critic found both missing facts — concrete, fixable notes, not a vague "make it better." That specificity is what makes the *next* step work.

## 2 · The Creator — revises using the critique

The Creator takes the current answer **plus the critique** and produces a better version. The critique is the whole point: without it, re-running the creator just gives another random draft; *with* it, the creator has a specific target.

In [ ]:
def creator(answer, critique):
    """Revise the answer to address the critique's issues."""
    if USE_FAKE:
        if not critique["issues"]:
            return answer
        issue = critique["issues"][0]                 # fix the top issue this round
        if "'3 days'" in issue:
            return answer.rstrip(". ") + ". You may work remotely up to 3 days per week."
        if "'manager approval'" in issue:
            return answer.rstrip(". ") + ", with manager approval."
        if "too long" in issue:
            return "You may work remotely up to 3 days per week, with manager approval."
        return answer
    return _real_creator(answer, critique)

def _real_creator(answer, critique):
    from openai import OpenAI
    client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
    prompt = (f"Question: {QUESTION}\n\nCurrent answer:\n{answer}\n\n"
              f"Fix these issues, keep it under {MAX_WORDS} words:\n- " + "\n- ".join(critique["issues"]))
    resp = client.chat.completions.create(model=CREATOR_MODEL, temperature=0.3,
        messages=[{"role": "user", "content": prompt}])
    return resp.choices[0].message.content.strip()

revised = creator(first_draft, c0)
print("revised:", revised)

One issue fixed — the "3 days" fact is now present. Notice the Creator fixed the *top* issue this round; the loop will come back for the rest. (A real creator often fixes several at once; fixing one-at-a-time here makes each round's progress crystal-clear.)

## 3 · The loop — Creator ↔ Critic until good enough

Now we alternate: critique → revise → critique → revise, stopping the moment the critic says **`is_good_enough`**. Watch the answer climb round by round.

In [ ]:
def critic_creator(question, draft, max_rounds=5):
    answer = draft
    for r in range(1, max_rounds + 1):
        critique = critic(answer)
        print(f"--- round {r} ---")
        print(f"  answer : {answer}")
        print(f"  issues : {critique['issues'] or 'none'}")
        if critique["is_good_enough"]:
            print("  --> converged (no issues left); stopping.\n")
            break
        answer = creator(answer, critique)
    return answer

final = critic_creator(QUESTION, first_draft)
print("FINAL:", final)

Trace what happened: the draft started vague and missing both facts; each round the critic named a specific gap and the creator closed it; once no issues remained, the critic returned `is_good_enough=True` and the loop **stopped on its own**. The final answer is materially better than the first draft — and you never wrote it by hand. That's systematic improvement.

## 4 · The convergence check — why `is_good_enough` matters

The single most important line in the loop is:

```python
if critique["is_good_enough"]:
    break
```

It means the loop stops when the answer is **actually done**, not after a fixed number of rounds. Two reasons this matters:
- **Cost.** Each round is 2 model calls (critic + creator). Stopping at convergence instead of always running 5 rounds can halve your spend.
- **Diminishing returns.** In practice **2–3 rounds is usually enough** — the big gains come early and flatten fast. Running 10 rounds rarely beats running 3.

Always pair the convergence check with a **`max_rounds` cap** (here, 5) as a safety net, so a critic that's never satisfied can't loop forever.

## 5 · You'll see this exact shape again

Critic-Creator is not a one-off trick — **two-prompt iteration is the skeleton of modern self-improving LLM systems.** You'll meet it repeatedly:
- **Week 11** — RAGAS runs an LLM-as-judge under the hood.
- **Week 17** — reflection loops (an agent critiques its own output).
- **Week 22** — plan-and-revise agents.
- **Week 28** — production evaluation pipelines.

Learn the shape now — *creator drafts, critic critiques against a rubric, creator revises, stop on convergence* — and you'll **recognize** it every time it reappears under a fancier name.

## 6 · Your turn — experiment

1. **Go live.** `USE_FAKE = False` + key. Real Creator (`gpt-4o-mini`) and Critic (`gpt-4o`). The real creator usually fixes multiple issues per round — so it converges *faster* than our one-at-a-time stand-in.
2. **Start from a *good* draft.** Feed in an already-solid answer and confirm the loop converges in **round 1** (the critic finds nothing) — proving it doesn't "improve" what's already fine.
3. **Add a rubric dimension.** Require a specific tone ("must be friendly") in the critic and watch a new kind of issue appear and get fixed.
4. **Break convergence.** Make the critic impossible to satisfy (require a fact that isn't true) and watch `max_rounds` save you from an infinite loop.

## Fit it into your app 🔧

This is the whole of **`src/eval/critic_creator.py`** — about 30 lines:
- `critic(answer)` — strong model (`gpt-4o`), returns issues + `is_good_enough`.
- `creator(answer, critique)` — cheap model (`gpt-4o-mini`), revises.
- `critic_creator(question, draft, max_rounds)` — the loop with the convergence check.

Lab Step 5: run `scripts/run_critic_creator.py` on a golden question where your `/ask` answer is weak, and save the round-by-round trace to `docs/critic-creator-trace.md`. It shows your answer *improving under a rubric* — strong evidence for Design Review 1.

**The one-line takeaway:** *don't just judge an answer — feed the critique back to a creator and loop; two prompts, a convergence check, and 2–3 rounds turn a weak draft into a good one, automatically.*